In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
from PIL import Image
import numpy as np
import cv2  # Для Connected Components и цветовой карты

In [3]:
class UnsupervisedVideoDataset(Dataset):
    def __init__(self, img_dir, img_size=(256, 256), device='cuda'):
        self.img_dir = img_dir
        self.img_size = img_size
        self.device = device

        # Надежный поиск файлов
        extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
        self.img_paths = []
        for ext in extensions:
            self.img_paths.extend(glob.glob(os.path.join(img_dir, ext)))
        self.img_paths = sorted(self.img_paths)

        if len(self.img_paths) < 2:
            raise ValueError(f"В папке '{img_dir}' должно быть минимум 2 изображения.")

        self.transform = T.Compose([
            T.Resize(self.img_size),
            T.ToTensor(),
        ])

        # Предобученный RAFT для подсчета оптического потока
        self.of_weights = Raft_Small_Weights.DEFAULT
        self.of_model = raft_small(weights=self.of_weights).to(device).eval()
        self.of_transforms = self.of_weights.transforms()

    def __len__(self):
        return len(self.img_paths) - 1

    @torch.no_grad()
    def _compute_optical_flow(self, img1_t, img2_t):
        img1_flow = img1_t.unsqueeze(0).to(self.device)
        img2_flow = img2_t.unsqueeze(0).to(self.device)
        img1_flow, img2_flow = self.of_transforms(img1_flow, img2_flow)

        list_of_flows = self.of_model(img1_flow, img2_flow)
        flow = list_of_flows[-1]
        return flow.squeeze(0).cpu()

    def __getitem__(self, idx):
        img1 = Image.open(self.img_paths[idx]).convert('RGB')
        img2 = Image.open(self.img_paths[idx + 1]).convert('RGB')

        img1_t = self.transform(img1)
        img2_t = self.transform(img2)

        flow_t = self._compute_optical_flow(img1_t, img2_t)

        # Вход сети: [5, H, W] (3 RGB + 2 Flow)
        x = torch.cat([img1_t, flow_t], dim=0)

        # На выходе: вход и поток (поток нужен для лосса)
        return x, flow_t

In [5]:
class MotionDetectorNet(nn.Module):
    def __init__(self):
        super(MotionDetectorNet, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(5, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.pool = nn.MaxPool2d(2, 2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU()
        )
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1)
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.pool(x1)
        x2 = self.enc2(x2)
        out = self.upsample(x2)
        out = self.dec(out)
        return torch.sigmoid(out)  # Маска от 0 до 1



In [7]:
def compute_unsupervised_loss(pred_mask, flow, lambda_smooth=0.1):
    # 1. Magnitude Loss: Движение => Маска
    flow_magnitude = torch.sqrt(flow[:, 0:1, :, :] ** 2 + flow[:, 1:2, :, :] ** 2 + 1e-8)
    max_val = torch.max(flow_magnitude.view(flow_magnitude.size(0), -1), dim=1)[0].view(-1, 1, 1, 1) + 1e-8
    target_motion_mask = flow_magnitude / max_val
    # Порог: отсекаем шум
    target_motion_mask = torch.where(target_motion_mask > 0.15, torch.ones_like(target_motion_mask),
                                     torch.zeros_like(target_motion_mask))

    magnitude_loss = F.mse_loss(pred_mask, target_motion_mask)

    # 2. Smoothness Loss: Гладкие края
    dy = pred_mask[:, :, 1:, :] - pred_mask[:, :, :-1, :]
    dx = pred_mask[:, :, :, 1:] - pred_mask[:, :, :, :-1]
    smoothness_loss = torch.mean(dy ** 2) + torch.mean(dx ** 2)

    return magnitude_loss + lambda_smooth * smoothness_loss

In [11]:
def get_instance_color_mask(binary_mask, threshold=0.6):
    """
    Пост-обработка бинарной маски движения в поэкземплярную цветную маску.
    Никакого обучения — чистая математика OpenCV.
    """
    # 1. Получаем numpy маску [H, W], значения от 0.0 до 1.0
    mask_np = binary_mask.squeeze().cpu().numpy()

    # 2. Применяем порог, чтобы получить строго бинарную маску [H, W], uint8
    mask_binary = (mask_np > threshold).astype(np.uint8)

    # 3. Устраняем мелкие шумы (Морфологическая операция открытия: Эрозия -> Дилатация)
    kernel = np.ones((5, 5), np.uint8)
    mask_clean = cv2.morphologyEx(mask_binary, cv2.MORPH_OPEN, kernel)

    # 4. Connected Components (Связные компоненты)
    # Находит все изолированные белые пятна и присваивает каждому ID: 0 (фон), 1, 2, ... N
    num_labels, labels_im = cv2.connectedComponents(mask_clean)

    # 5. Генерация цветной карты
    if num_labels <= 1:
        # Объектов нет, возвращаем черный квадрат
        return np.zeros((labels_im.shape[0], labels_im.shape[1], 3), dtype=np.uint8), 0

    # Создаем карту цветов. Сдвигаем ID на 1, чтобы фон остался черным, а первый объект был ярким.
    # labels_im_normalized = 255.0 * (labels_im / num_labels)
    # labels_im_normalized = labels_im_normalized.astype(np.uint8)

    # Более надежный способ сделать разные цвета для разных ID
    colors = []
    # Цвет для фона (черный)
    colors.append((0, 0, 0))
    # Цвета для объектов
    np.random.seed(42)  # Чтобы цвета были одинаковые от запуска к запуску
    for i in range(1, num_labels):
        colors.append((np.random.randint(50, 256), np.random.randint(50, 256), np.random.randint(50, 256)))

    color_mask = np.zeros((labels_im.shape[0], labels_im.shape[1], 3), dtype=np.uint8)
    for r in range(labels_im.shape[0]):
        for c in range(labels_im.shape[1]):
            color_mask[r, c] = colors[labels_im[r, c]]

    # Если нам нужно наложить цвета на фон (ColorMap), OpenCV предоставляет для этого функции
    # color_map = cv2.applyColorMap(labels_im_normalized, cv2.COLORMAP_JET)
    # Игнорируем фон: если исходная маска 0, ставим 0
    # color_map[mask_clean == 0] = 0

    return color_mask, num_labels - 1

In [13]:
def overlay_mask_on_image(image_t, color_mask, alpha=0.6):
    """Наложение полупрозрачной цветной маски на исходный кадр"""
    img_np = image_t.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * 255).astype(np.uint8)
    img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)  # OpenCV работает в BGR

    overlay = cv2.addWeighted(img_np, 1.0, color_mask, alpha, 0)

    # Добавляем подпись
    cv2.putText(overlay, f"Instance Overlay", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return overlay

In [19]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_DIR = "clean_1008fps\cave_2"  # Укажите путь к вашей папке с последовательными кадрами
BATCH_SIZE = 4
EPOCHS = 20  # Нужно больше эпох для обучения без учителя
LEARNING_RATE = 3e-4

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\anna\AppData\Local\Temp\ipykernel_33788\91619450.py:2: SyntaxWarning: invalid escape sequence '\c'
  IMG_DIR = "clean_1008fps\cave_2"  # Укажите путь к вашей папке с последовательными кадрами


In [23]:
dataset = UnsupervisedVideoDataset(img_dir=IMG_DIR, img_size=(256, 256), device=DEVICE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [25]:
model = MotionDetectorNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
print("Начало обучения на бинарную маску движения (без учителя)...")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for inputs, flows in dataloader:
        inputs = inputs.to(DEVICE)
        flows = flows.to(DEVICE)

        pred_masks = model(inputs)
        loss = compute_unsupervised_loss(pred_masks, flows, lambda_smooth=0.08)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Эпоха [{epoch + 1}/{EPOCHS}] - Loss: {epoch_loss / len(dataloader):.4f}")

torch.save(model.state_dict(), "unsupervised_motion_instance_detector.pth")
print("Обучение завершено. Модель сохранена.")

# 4. ИНФЕРЕНС И ВИЗУАЛИЗАЦИЯ INSTANCE SEGMENTATION (Поэкземплярной)
print("\nНачинаем предсказание и визуализацию экземпляров...")
model.eval()